INSTALL THE REQUIRED PACKAGES

In [1]:
%pip -q install -U langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


CONFIGURE YOUR OPEN_API_KEY

In [2]:
import os
from getpass import getpass

if not os.environ.get("OPEN_API_KEY"):
  os.environ["OPEN_API_KEY"] = getpass("Enter your open API key")

print("API KEY CONFIRMED:-",bool(os.environ.get("OPEN_API_KEY")))

Enter your open API key··········
API KEY CONFIRMED:- True


Uploading the travel policy document to collab and setting its path


---



In [3]:
from pathlib import Path
from google.colab import files

POLICY_PATH = Path("/content/Demo_Corporate_Travel_Policy.txt")


print("USING THE DOCUMENT:-",POLICY_PATH.name)
print("FILE SIZE:-",POLICY_PATH.stat().st_size,"bytes")

USING THE DOCUMENT:- Demo_Corporate_Travel_Policy.txt
FILE SIZE:- 11997 bytes


Loading the document and splitting into chunks

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents=TextLoader(
    str(POLICY_PATH),
    encoding="utf-8"
).load()

#assigning the filename its source
for document in documents:
  document.metadata["SOURCE"]=POLICY_PATH.name

splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
chunks=splitter.split_documents(documents)

print("Documents loaded:", len(documents))
print("Chunks created:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:500])

/tmp/ipykernel_885/3258225531.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Documents loaded: 1
Chunks created: 17

First chunk preview:

DEMO CORPORATE TRAVEL POLICY

Document Type: Internal Corporate Policy
Policy Owner: Human Resources and Administration
Version: 1.0
Effective Date: 1 January 2026
Review Cycle: Annual
Applicability: All employees and approved contractors

IMPORTANT
---------
This is a fictional demonstration policy created for a Retrieval-Augmented
Generation (RAG) chatbot workshop. It is not a real company's policy and should
not be used to approve actual travel or expenses.

1


CREATE EMBEDDINGS AND FAISS VECTORBASE

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)                                       # take each item in documents
                                        # turn its text into a vector using embedding
                                        # store everything

retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

print("FAISS vector store sucessfully created")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model="gpt-40-mini",
    temperature=0
)

prompt = ChatPromptTemplate.from_template(
    """You are a helpful corporate Travel Desk assistant.

Answer the employee's question using ONLY the policy context below.
Do not invent policy rules, reimbursement amounts, vendors, or approvals.
If the policy does not contain the answer, say:
"I couldn't find this information in the travel policy. Please contact the travel desk."

Policy context:
{context}

Employee question:
{question}

Answer clearly and concisely:"""
)
def format_context(retrieved_docs):
  return "\n\n".join(
      doc.page_content for doc in retrieved_docs
  )


def ask_travel_desk(question):
  #retrieval relevant policy chunks
  retrieved_docs=retriever.invoke(question)
  #prpare context
  context=format_context(retieved_docs)
  #send context + question to llm
  answer=(prompt|llm|StrOutParser()).invoke({
      "context":context,
      "question":question
  })
  #Display of answers
  print("\nTRAVEL DESK CHATBOT")
  print("-" * 55)
  print(answer)

  print("\nSOURCES:")
  file_names = set()                          # 1. make an empty set

  for doc in retrieved_docs:                  # 2. go through each chunk
      name = doc.metadata.get("source", "Unknown")   # 3. get its file name
      file_names.add(name)                    # 4. add it (duplicates are ignored)

  for source in sorted(file_names):           # 5. go through the names A→Z
      print("-", source)                      # 6. print each one

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

Ask a question

In [ ]:
question = "What should I do if my flight gets cancelled?"
ask_travel_desk(question)

TRY MORE QUESTIONS

In [9]:
test_questions = [
    "Do I need approval before booking a domestic flight?",
    "Which cabin class should I book?",
    "Does the policy specify a hotel reimbursement amount?",
    "What documents are required for an expense claim?",
    "What should I do if my flight gets cancelled?",
    "Does the travel desk guarantee visa approval?",
    "Are personal sightseeing expenses reimbursable?"
]

for question in test_questions:
  print("\n"+ "-" * 60)
  print("Question:", question)
  ask_travel_desk(question)


------------------------------------------------------------
Question: Do I need approval before booking a domestic flight?


NameError: name 'ask_travel_desk' is not defined